In [1]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
from copy import deepcopy
from copy import deepcopy

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5
from copy import deepcopy

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml as read_snap_xml

# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)
# pg.setConfigOptions(useOpenGL=False)   # 若驱动或 OpenGL 有问题可显式关闭
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

from config import DATA_DIR,INPUT_DIR


backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [2]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [start, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 30152
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)


In [10]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml
start_ts =20822

end_ts = 20865

# 可以选择，是看处理后的，还是原始的
# file_path = INPUT_DIR / f"onestep/interplane_links_{start_ts}_{end_ts}.xml"
# rawnodes = write2xml.xml_to_nodes2(file_path, tegnode.tegnode)

file_path = INPUT_DIR / f"interplane_links_{start_ts}_{end_ts}.xml"
rawnodes = write2xml.xml_to_nodes(file_path, tegnode.tegnode)


# rawnodes  = write2xml.xml_to_nodes(  rf"E:\研究生\研究进展\工作记录\实验记录\interplane_links_{start_ts}_{end_ts}.xml", tegnode.tegnode)

# 这里，我们读取到nodes 信息,但是我们需要转化为edge信息，注意这里主要就只有包括inter-edge信息


In [11]:

group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [12]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 注意 ，下面是直接将nodes转为edge，因为我们的nodes本身已经完成了冲突检测和处理
import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_inter_edge = inter_edge2nodes.trans_nodes2edges(rawnodes,P,N)


In [6]:
# 转化为inter-edge信息后，我们可以通过绘图来初步查看

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


In [13]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("raw behand")
viewer.resize(1200, 700)
viewer.edges_by_step = all_inter_edge

viewer.show()

